In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime, timezone

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2_DIR = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE3_DIR = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_DIR.mkdir(parents=True, exist_ok=True)

# Master scales (append/replace per subject + p)
PHASE3_MASTER_PATH = PHASE3_DIR / "03_ENV_SCALE_ALL_SUBJECTS.csv"

# Optional: per-subject report of problematic sensors
PHASE3_BADSCALES_DIR = PHASE3_DIR / "bad_scales_reports"
PHASE3_BADSCALES_DIR.mkdir(parents=True, exist_ok=True)

print("PHASE2_DIR:", PHASE2_DIR)
print("PHASE3_DIR:", PHASE3_DIR)
print("MASTER   :", PHASE3_MASTER_PATH)
print("BAD DIR  :", PHASE3_BADSCALES_DIR)


In [ ]:
def load_phase2_subject(subject_name: str, columns=None) -> pd.DataFrame:
    """
    Loads Phase 2 parquet for a subject.
    Only required columns are read to save RAM.
    """
    path = PHASE2_DIR / f"{subject_name}__emg_bp_env.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Phase 2 parquet not found: {path}")

    if columns is None:
        columns = ["subject", "trial_id", "sensor", "env"]

    df = pd.read_parquet(path, columns=columns)

    needed = {"subject", "trial_id", "sensor", "env"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Phase 2 parquet missing columns: {missing}")

    # Basic clean: keep finite env only (should already be ok in your pipeline)
    df = df.copy()
    df["subject"] = df["subject"].astype(str)
    df["trial_id"] = df["trial_id"].astype(str)
    df["sensor"] = df["sensor"].astype(str)

    return df


In [ ]:
def sample_for_percentile(
    df_subj: pd.DataFrame,
    mode: str = "random_per_sensor",
    frac: float = 0.02,
    min_per_sensor: int = 5_000,
    max_per_sensor: int = 200_000,
    random_state: int = 0,
) -> pd.DataFrame:
    """
    Sampling to estimate percentile scale.

    mode:
      - "none": use full data (can be heavy)
      - "random_global": random sample over all rows
      - "random_per_sensor": sample within each (subject,sensor) group (recommended)

    Notes:
      - env is clipped to >= 0 (avoid tiny negatives from filtfilt)
      - keeps only finite env values for scale estimation
    """
    df = df_subj.copy()

    env = df["env"].to_numpy(dtype=float)
    # keep only finite env for computing percentile scale
    finite_mask = np.isfinite(env)
    df = df.loc[finite_mask].copy()

    # clip tiny negatives
    df["env_clip"] = np.clip(df["env"].to_numpy(dtype=float), 0.0, None)

    if mode == "none":
        return df

    if mode == "random_global":
        n = int(round(len(df) * frac))
        n = max(1, min(n, len(df)))
        return df.sample(n=n, random_state=random_state)

    if mode == "random_per_sensor":
        def _sample_group(g: pd.DataFrame) -> pd.DataFrame:
            n = int(round(len(g) * frac))
            n = max(min_per_sensor, n)
            n = min(n, len(g), max_per_sensor)
            return g.sample(n=n, random_state=random_state) if n < len(g) else g

        return (
            df.groupby(["subject", "sensor"], group_keys=False)
              .apply(_sample_group)
              .reset_index(drop=True)
        )

    raise ValueError("mode must be one of: none, random_per_sensor, random_global")


In [ ]:
def compute_percentile_scales(
    df_sampled: pd.DataFrame,
    p: int = 95,
    too_small_thr: float = 1e-8,
) -> pd.DataFrame:
    """
    Output schema:
      subject | sensor | p | env_scale | scale_too_small_flag | n_used_for_scale
    Policy:
      - If env_scale < too_small_thr OR env_scale is NaN -> scale_too_small_flag=True
      - If too_small -> env_scale is set to NaN (so normalization won't silently explode)
    """
    if "env_clip" not in df_sampled.columns:
        raise ValueError("env_clip column not found. Call sample_for_percentile first.")

    q = p / 100.0

    # compute quantile per (subject,sensor)
    scales = (
        df_sampled
        .groupby(["subject", "sensor"], as_index=False)
        .agg(
            env_scale=("env_clip", lambda x: float(np.quantile(x.to_numpy(), q)) if len(x) else np.nan),
            n_used_for_scale=("env_clip", "size"),
        )
    )

    scales["p"] = int(p)

    # clean zero -> NaN
    scales["env_scale"] = scales["env_scale"].replace(0.0, np.nan)

    # policy flag
    scales["scale_too_small_flag"] = (
        scales["env_scale"].isna() | (scales["env_scale"] < too_small_thr)
    )

    # enforce policy: if too small -> set to NaN
    scales.loc[scales["scale_too_small_flag"], "env_scale"] = np.nan

    return scales[["subject", "sensor", "p", "env_scale", "scale_too_small_flag", "n_used_for_scale"]]


In [ ]:
def upsert_subject_p_in_master(
    subject_name: str,
    p: int,
    new_scales: pd.DataFrame,
    master_path: Path,
    mode: str = "append",   # "append" or "replace"
    meta: dict | None = None,
):
    """
    Master schema (recommended):
      subject, sensor, p, env_scale, scale_too_small_flag, n_used_for_scale,
      sampling_mode, frac, min_per_sensor, max_per_sensor, random_state,
      phase2_rows, phase2_trials, computed_utc

    Unique logical key: (subject, p, sensor)
    mode:
      - append: only add if (subject,p) does not exist at all
      - replace: remove existing rows for (subject,p) then add new
    """
    if new_scales.empty:
        print(f"No rows to write for {subject_name}, p={p}.")
        return

    df_new = new_scales.copy()
    df_new["subject"] = df_new["subject"].astype(str)
    df_new["sensor"] = df_new["sensor"].astype(str)
    df_new["p"] = df_new["p"].astype(int)

    # attach metadata columns (same value for all rows)
    if meta:
        for k, v in meta.items():
            df_new[k] = v

    if master_path.exists():
        old = pd.read_csv(master_path)
        old["subject"] = old["subject"].astype(str)
        old["sensor"] = old["sensor"].astype(str)
        old["p"] = old["p"].astype(int)
        exists_sp = ((old["subject"] == str(subject_name)) & (old["p"] == int(p))).any()
    else:
        old = pd.DataFrame()
        exists_sp = False

    if mode == "append":
        if exists_sp:
            print(f"(append) ({subject_name}, p={p}) already exists in {master_path.name}. Skipping.")
            return
        combined = pd.concat([old, df_new], ignore_index=True) if len(old) else df_new

    elif mode == "replace":
        if len(old):
            kept = old[~((old["subject"] == str(subject_name)) & (old["p"] == int(p)))].copy()
            combined = pd.concat([kept, df_new], ignore_index=True)
        else:
            combined = df_new

    else:
        raise ValueError("mode must be 'append' or 'replace'")

    master_path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(master_path, index=False)
    print(f"Wrote master scales -> {master_path} | rows now: {len(combined)}")


In [ ]:
def save_bad_scales_report(subject_name: str, p: int, scales: pd.DataFrame) -> Path | None:
    bad = scales[scales["scale_too_small_flag"] == True].copy()
    if bad.empty:
        print("No bad scales detected.")
        return None

    out = PHASE3_BADSCALES_DIR / f"{subject_name}__p{p}__bad_scales.csv"
    bad.to_csv(out, index=False)
    print("Saved bad scales report:", out)
    return out


In [ ]:
SUBJECT_NAME = "ALS_Subject_1"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_2"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_3"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_4"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_5"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_7"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_8"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_9"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_10"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_11"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_12"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_13"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_14"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_15"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "ALS_Subject_16"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_1"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_2"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_3"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_4"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_5"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_6"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_7"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
SUBJECT_NAME = "Healthy_Subject_8"   # change only this
P = 95

MODE = "append"    # for a new subject
# MODE = "replace" # if re-processing a subject

SAMPLING_MODE = "random_per_sensor"   # recommended
FRAC = 0.02
MIN_PER_SENSOR = 5_000
MAX_PER_SENSOR = 200_000
RANDOM_STATE = 0

TOO_SMALL_THR = 1e-8  # policy threshold

# 1) Load phase 2 output (minimal cols)
df_subj = load_phase2_subject(SUBJECT_NAME, columns=["subject","trial_id","sensor","env"])
phase2_rows = int(len(df_subj))
phase2_trials = int(df_subj["trial_id"].nunique())
phase2_sensors = int(df_subj["sensor"].nunique())
print("Loaded rows:", phase2_rows, "| trials:", phase2_trials, "| sensors:", phase2_sensors)

# 2) Sample
df_s = sample_for_percentile(
    df_subj,
    mode=SAMPLING_MODE,
    frac=FRAC,
    min_per_sensor=MIN_PER_SENSOR,
    max_per_sensor=MAX_PER_SENSOR,
    random_state=RANDOM_STATE
)
print("Sampled rows:", len(df_s))

# 3) Compute Pth percentile scales
scales = compute_percentile_scales(df_s, p=P, too_small_thr=TOO_SMALL_THR)
display(scales.sort_values(["sensor"]))

# 4) Save bad-scale report (if any)
save_bad_scales_report(SUBJECT_NAME, P, scales)

# 5) Upsert into master with meta
meta = {
    "sampling_mode": SAMPLING_MODE,
    "frac": float(FRAC),
    "min_per_sensor": int(MIN_PER_SENSOR),
    "max_per_sensor": int(MAX_PER_SENSOR),
    "random_state": int(RANDOM_STATE),
    "phase2_rows": phase2_rows,
    "phase2_trials": phase2_trials,
    "computed_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
upsert_subject_p_in_master(
    subject_name=SUBJECT_NAME,
    p=P,
    new_scales=scales,
    master_path=PHASE3_MASTER_PATH,
    mode=MODE,
    meta=meta
)


In [ ]:
# first make sure everything is imported and defined
import pandas as pd
import numpy as np
from pathlib import Path

# define again if not defined yet (or copy from the first cell)
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2_DIR = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE3_DIR = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_MASTER_PATH = PHASE3_DIR / "03_ENV_SCALE_ALL_SUBJECTS.csv"

PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"
PHASE3_NORM_DIR.mkdir(parents=True, exist_ok=True)

# now the main cell
if not PHASE3_MASTER_PATH.exists():
    print("Master file not found! Run Phase 3 for at least one subject first.")
else:
    scales_master = pd.read_csv(PHASE3_MASTER_PATH)
    scales_valid = scales_master[
        (scales_master["p"] == 95) &
        (~scales_master["scale_too_small_flag"]) &
        (scales_master["env_scale"].notna())
    ].copy()
    
    all_subjects = sorted(scales_valid["subject"].unique())
    print(f"Subjects to process: {len(all_subjects)}")
    
    for subject in all_subjects:
        print(f"\n-> processing {subject}")
        
        path_phase2 = PHASE2_DIR / f"{subject}__emg_bp_env.parquet"
        if not path_phase2.exists():
            print("  Phase-2 file not found -> skipped")
            continue
        
        try:
            df = pd.read_parquet(path_phase2)
        except Exception as e:
            print(f"  error reading file: {e}")
            continue
        
        scales_sub = scales_valid[scales_valid["subject"] == subject]
        if scales_sub.empty:
            print("  no valid scale for this subject -> skipped")
            continue
        
        df_merged = df.merge(
            scales_sub[["sensor", "env_scale"]],
            on="sensor",
            how="left"
        )
        
        df_merged["env_norm"] = np.where(
            df_merged["env_scale"].notna(),
            df_merged["env"] / df_merged["env_scale"],
            np.nan
        )
        
        out_path = PHASE3_NORM_DIR / f"{subject}__emg_env_norm.parquet"
        df_merged.to_parquet(out_path, index=False)
        print(f"  saved: {out_path} | rows: {len(df_merged):,}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PHASE2_DIR = PROCESSED_ROOT / "phase_02_preprocess_emg"
PHASE3_DIR = PROCESSED_ROOT / "phase_03_normalization"
PHASE3_MASTER_PATH = PHASE3_DIR / "03_ENV_SCALE_ALL_SUBJECTS.csv"

PHASE3_NORM_DIR = PHASE3_DIR / "normalized_env_per_subject"
PHASE3_NORM_DIR.mkdir(parents=True, exist_ok=True)

# ===== change only this =====
SUBJECT_NAME = "Healthy_Subject_8"
P = 95

# 1) load the master scale table
if not PHASE3_MASTER_PATH.exists():
    raise FileNotFoundError("Master file not found. Compute and upsert the scale for this subject first.")

scales_master = pd.read_csv(PHASE3_MASTER_PATH)

# 2) valid scales of this subject
scales_sub = scales_master[
    (scales_master["subject"].astype(str) == str(SUBJECT_NAME)) &
    (scales_master["p"].astype(int) == int(P)) &
    (~scales_master["scale_too_small_flag"]) &
    (scales_master["env_scale"].notna())
].copy()

if scales_sub.empty:
    raise ValueError(f"No valid scale for {SUBJECT_NAME} in the master table (p={P}).")

# 3) load the Phase-2 file of this subject
path_phase2 = PHASE2_DIR / f"{SUBJECT_NAME}__emg_bp_env.parquet"
if not path_phase2.exists():
    raise FileNotFoundError(f"Phase-2 file not found: {path_phase2}")

df = pd.read_parquet(path_phase2)

# 4) merge + normalize
df_merged = df.merge(scales_sub[["sensor", "env_scale"]], on="sensor", how="left")
df_merged["env_norm"] = np.where(df_merged["env_scale"].notna(), df_merged["env"] / df_merged["env_scale"], np.nan)

# (optional) sensor coverage check
coverage = df_merged["env_scale"].notna().mean()
print(f"Coverage (rows with scale): {coverage:.2%}")

# 5) save (this subject only)
out_path = PHASE3_NORM_DIR / f"{SUBJECT_NAME}__emg_env_norm.parquet"
df_merged.to_parquet(out_path, index=False)
print(f"Saved: {out_path} | rows: {len(df_merged):,}")
